In [1]:
import torch
import numpy as np
import json
# import scipy.special as sp

# import pickle as pkl
# import zlib
# import base64

In [2]:
import sys
sys.path.append('/home/kutulu/projects/code-of-kutulu-client')

In [3]:
from src.envs.agents.reinforce_agent import REINFORCEAgent
from src.envs.agents.dqn_agent_ext import DQNAgentExt
from src.envs.agents.dqn_agent import DQNAgent

In [9]:
agents_dir = '../output/2025-05-28/20250528-122750'
agent_id = 0

In [84]:
checkpoint_dir = f'{agents_dir}/agent{agent_id}/5000'

In [85]:
with open(f'{agents_dir}/agents_info.json') as f:
    agents_info = json.load(f)
    info = agents_info[agent_id]
    del info['type']

In [86]:
agent = DQNAgentExt(**info)
agent.train = False

In [87]:
agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

<All keys matched successfully>

In [88]:
# agent = REINFORCEAgent(**{
#     'state_type': 'closest_ext',
#     'gamma': 0.5,
#     'action_space_n': 8,
#     'train': False,
# })

In [89]:

from src.envs.distance import find_path
from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS
from src.game.template import MOVE_REL_POS, REL_POSITIONS

In [90]:
env = KutuluWorldEnv('', '', 1, actions=EXTENDED_KUTULU_ACTIONS)
env.map = [
    #0123456
    '#######', # 0
    '#.....#', # 1
    '#.#.#.#', # 2
    '#.....#', # 3
    '#.#.#.#', # 4
    '#.....#', # 5
    '#######', # 6
]
env.width = len(env.map[0])
env.height = len(env.map)

In [91]:
all_answers = set(range(4))
params = [
    (set([answer]), [rel_pos], [])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (set([answer]), [(tuple(x * 2 for x in rel_pos))], [])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (set([answer]), [(tuple(x * 3 for x in rel_pos))], [])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (all_answers - set([answer]), [], [rel_pos])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (all_answers - set([answer]), [], [tuple(x * 2 for x in rel_pos)])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
]

In [92]:
def set_env(agent, env, explorers, wanderers):
    player_id = 0
    player_pos = (3, 3)
    obs = [
        None,
        f'EXPLORER 0 {player_pos[0]} {player_pos[1]} 100 0 0'
    ] + [
        f'EXPLORER {i + 1} {player_pos[0] + x} {player_pos[1] + y} 10 0 0'
        for i, (x, y) in enumerate(explorers) 
    ] + [
        f'WANDERER {i + 10} {player_pos[0] + x} {player_pos[1] + y} 10 1 0'
        for i, (x, y) in enumerate(wanderers) 
    ]
    env._set_entities(obs)
    env._set_players(obs, set_ids=True)
    agent.set_env(env)

In [93]:
for answer, explorers, wanderers in params:
    set_env(agent, env, explorers, wanderers)
    state, action = agent.generate_state_and_step(0)
    print(action, answer, explorers, wanderers)

0 {0} [(0, -1)] []
0 {1} [(1, 0)] []
0 {2} [(0, 1)] []
0 {3} [(-1, 0)] []
0 {0} [(0, -2)] []
2 {1} [(2, 0)] []
0 {2} [(0, 2)] []
2 {3} [(-2, 0)] []
0 {0} [(0, -3)] []
0 {1} [(3, 0)] []
0 {2} [(0, 3)] []
0 {3} [(-3, 0)] []
1 {1, 2, 3} [] [(0, -1)]
2 {0, 2, 3} [] [(1, 0)]
0 {0, 1, 3} [] [(0, 1)]
1 {0, 1, 2} [] [(-1, 0)]
1 {1, 2, 3} [] [(0, -2)]
2 {0, 2, 3} [] [(2, 0)]
0 {0, 1, 3} [] [(0, 2)]
2 {0, 1, 2} [] [(-2, 0)]


In [94]:
answer, explorers, wanderers = params[1]
set_env(agent, env, explorers, wanderers)
state, action = agent.generate_state_and_step(0)

In [95]:
action, answer, explorers, wanderers

(np.int64(0), {1}, [(1, 0)], [])

In [96]:
self = agent
player_id = 0

In [97]:

self.frame_idx += 1

valid_actions = self.get_valid_actions(player_id)
player_mask = ~np.array(valid_actions)
player_mask = player_mask[:self.action_space_n]

state = self.get_state(player_id)
data = self.episode_buffer.encode_states([state])

In [98]:
self = self.model

In [99]:
assert data['entity_dir'].shape[-1] == self.num_dirs

x_kind_embs = self.kind_embs(data['entity_kind'])

entity_features = data['entity_features']
x_features = self.features_linear(entity_features)

entity_dir = data['entity_dir']
x_dir = self.dir_linear(entity_dir)
entities_mask = (entity_dir > 0).max(dim=-1, keepdim=True)[0]

# [batch_size, entity_dim, embed_dim + hidden_dim + inner_dim]
x_entitity = torch.cat((x_kind_embs, x_features, x_dir), dim=-1)
# [batch_size, entity_dim, inner_dim]
x = self.entity_linear(x_entitity)

# entity_weights = self.entity_impact(x_entitity) * entities_mask
entity_weights = torch.softmax(self.entity_impact(x_entitity), dim=-1) * entities_mask
# [batch_size, inner_dim, entity_dim]
x_transposed = x.transpose(1, 2)
# [batch_size, inner_dim, num_classes]
x = torch.bmm(x_transposed, entity_weights)
# [batch_size, num_classes, inner_dim]
x = x.transpose(2, 1)
# [batch_size, num_classes]
output = self.out_linear(x.reshape(-1, self.num_classes * self.inner_dim))

In [102]:
output

tensor([[ 0.0073,  0.0061,  0.0068,  0.0057,  0.0641,  0.0180,  0.0106, -0.0343]],
       grad_fn=<AddmmBackward0>)

In [105]:
x_entitity

tensor([[[-0.6977, -2.1059, -0.1461,  ...,  0.5730,  0.0271, -0.1460],
         [ 1.5836,  0.2096,  1.6686,  ...,  0.2610,  0.1146, -0.1244],
         [ 1.5836,  0.2096,  1.6686,  ...,  0.2610,  0.1146, -0.1244],
         ...,
         [ 1.5836,  0.2096,  1.6686,  ...,  0.2610,  0.1146, -0.1244],
         [ 1.5836,  0.2096,  1.6686,  ...,  0.2610,  0.1146, -0.1244],
         [ 1.5836,  0.2096,  1.6686,  ...,  0.2610,  0.1146, -0.1244]]],
       grad_fn=<CatBackward0>)

In [104]:
self.entity_impact(x_entitity)

tensor([[[  1.8449, -10.8001,  -0.9138,   1.8049,  -2.2458,   1.1164,   2.8130,
            1.5953],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           -0.4728],
         [ -0.3742,  -0.0887,  -0.5627,  -0.2143,  -0.4218,  -0.2570,  -0.1607,
           

In [103]:
entity_weights

tensor([[[1.6847e-01, 5.4308e-07, 1.0676e-02, 1.6187e-01, 2.8181e-03,
          8.1308e-02, 4.4360e-01, 1.3127e-01],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000

In [46]:
model_output = output[0].detach().cpu().numpy()

In [47]:
actions_masked = np.ma.array(model_output, mask=player_mask)

In [48]:
actions_masked

masked_array(data=[0.0031038778834044933, 0.0053454115986824036,
                   0.0010488568805158138, 0.0024201711639761925, --, --,
                   --, --],
             mask=[False, False, False, False,  True,  True,  True,  True],
       fill_value=np.float64(1e+20),
            dtype=float32)

In [49]:
self.train

<bound method Module.train of DQNExt(
  (kind_embs): Embedding(13, 32)
  (features_linear): Linear(in_features=7, out_features=32, bias=True)
  (dir_linear): Linear(in_features=5, out_features=16, bias=True)
  (entity_linear): Linear(in_features=80, out_features=16, bias=True)
  (entity_impact): Linear(in_features=80, out_features=8, bias=True)
  (out_linear): Linear(in_features=128, out_features=8, bias=True)
)>

In [30]:
action = actions_masked.argmax()

In [31]:
action

np.int64(1)

In [34]:
import torch
import torch.nn as nn
out_linear = nn.Linear(16 * 8, 8)

In [35]:
out_linear(x.reshape(-1, 16 * 8))

tensor([[-0.0682,  0.0097,  0.0179, -0.1315,  0.0023, -0.1417,  0.0012, -0.0833]],
       grad_fn=<AddmmBackward0>)

In [130]:
16 * 8 * 8

1024

In [ ]:

# [batch_size, num_classes]
output = self.out_linear(x).squeeze(-1)

In [115]:
output

tensor([[13.3777, 11.4812,  9.7850, 13.0480, -1.5532,  3.6007,  6.6069, -2.4162]],
       grad_fn=<SqueezeBackward1>)

In [108]:
output

tensor([[12.6539, 10.6024,  8.9396, 12.1601, -1.5361,  3.0114,  6.0064, -2.9908]],
       grad_fn=<SqueezeBackward1>)

In [16]:
set_env(agent, env, [(0, -2)], [])

In [17]:
self = agent

In [18]:
player_id = 0

In [19]:
valid_actions = self.get_valid_actions(player_id)
player_mask = ~np.array(valid_actions)
player_mask = player_mask[:self.action_space_n]


state = self.get_state(player_id)
data = self.episode_buffer.encode_states([state])
model_output = self.model(data)[0].detach().cpu().numpy()
actions_masked = np.ma.array(model_output, mask=player_mask)

In [20]:
model_output.std()

np.float32(0.04868661)

In [21]:
model_output

array([-0.31248164, -0.23298003, -0.32872823, -0.21628943], dtype=float32)

In [22]:
actions_masked.std()

np.float64(0.04868661068708801)

In [23]:
(actions_masked / actions_masked.sum()).std()

np.float64(0.0446469902019552)